In [0]:
USE CATALOG big_data_team1;
USE SCHEMA mathias;

-- ============================================================================
-- CONSULTA 1: Vista preliminar de datos clave (Muestreo representativo)
-- En lugar de un SELECT *, mostramos únicamente columnas analíticas relevantes.
-- ============================================================================
SELECT 
  codigo_inventario,
  departamento,
  provincia,
  distrito,
  tipo_peligro,
  nombre_peligro,
  anio_registro,
  viviendas_afectadas,
  personas_damnificadas
FROM big_data_team1.mathias.ingemmet_silver 
WHERE anio_registro IS NOT NULL
ORDER BY fecha_registro DESC
LIMIT 10;


-- ============================================================================
-- CONSULTA 2: Dashboard general de métricas clave (KPIs Globales)
-- Muestra el panorama total del dataset en un solo bloque consolidado.
-- ============================================================================
SELECT 
  COUNT(*) AS total_eventos_registrados,
  COUNT(DISTINCT departamento) AS total_departamentos_afectados,
  COUNT(DISTINCT tipo_peligro) AS tipos_peligro_identificados,
  COALESCE(SUM(viviendas_afectadas), 0) AS total_viviendas_afectadas,
  COALESCE(SUM(personas_damnificadas), 0) AS total_personas_damnificadas,
  COALESCE(SUM(personas_fallecidas), 0) AS total_personas_fallecidas
FROM big_data_team1.mathias.ingemmet_silver;


-- ============================================================================
-- CONSULTA 3: Top 5 Departamentos con mayor volumen de eventos y % del total
-- Calcula la participación porcentual de cada departamento frente al total nacional.
-- ============================================================================
SELECT 
  departamento, 
  COUNT(*) AS total_eventos,
  ROUND((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER()), 2) AS porcentaje_del_total
FROM big_data_team1.mathias.ingemmet_silver
WHERE departamento IS NOT NULL
GROUP BY departamento
ORDER BY total_eventos DESC
LIMIT 5;


-- ============================================================================
-- CONSULTA 4: Frecuencia e Impacto Humano por Tipo de Peligro
-- Relaciona el tipo de amenaza geológica con el promedio y acumulado de daños.
-- ============================================================================
SELECT 
  tipo_peligro, 
  COUNT(*) AS total_eventos,
  SUM(viviendas_afectadas) AS viviendas_afectadas,
  SUM(personas_damnificadas) AS personas_damnificadas,
  ROUND(AVG(personas_damnificadas), 1) AS promedio_damnificados_por_evento
FROM big_data_team1.mathias.ingemmet_silver
WHERE tipo_peligro IS NOT NULL
GROUP BY tipo_peligro
ORDER BY total_eventos DESC;


-- ============================================================================
-- CONSULTA 5: Evaluación de Impacto Socioeconómico por Departamento
-- Ranking territorial según el grado de severidad y pérdidas materiales/humanas.
-- ============================================================================
SELECT 
  departamento,
  COUNT(*) AS total_eventos,
  SUM(viviendas_afectadas) AS viviendas_afectadas,
  SUM(viviendas_destruidas) AS viviendas_destruidas,
  SUM(personas_damnificadas) AS personas_damnificadas,
  SUM(personas_fallecidas) AS personas_fallecidas
FROM big_data_team1.mathias.ingemmet_silver
WHERE departamento IS NOT NULL
GROUP BY departamento
ORDER BY personas_damnificadas DESC, viviendas_afectadas DESC;


-- ============================================================================
-- CONSULTA 6: Análisis Histórico Comparativo de Riesgo por Año
-- Permite identificar las épocas de mayor siniestralidad o impacto ambiental.
-- ============================================================================
SELECT 
  anio_registro AS anio,
  COUNT(*) AS total_eventos,
  SUM(viviendas_afectadas) AS viviendas_afectadas,
  SUM(personas_damnificadas) AS personas_damnificadas,
  SUM(personas_fallecidas) AS personas_fallecidas
FROM big_data_team1.mathias.ingemmet_silver
WHERE anio_registro IS NOT NULL AND anio_registro > 1900
GROUP BY anio_registro
ORDER BY personas_damnificadas DESC, viviendas_afectadas DESC;


-- ============================================================================
-- CONSULTA 7: Top 10 Distritos con Mayor Nivel de Severidad Humana
-- Análisis a nivel micro-territorial (Localidades más vulnerables).
-- ============================================================================
SELECT 
  departamento,
  provincia,
  distrito,
  COUNT(*) AS total_eventos,
  SUM(personas_damnificadas) AS total_damnificados,
  SUM(viviendas_afectadas) AS total_viviendas_afectadas
FROM big_data_team1.mathias.ingemmet_silver
WHERE distrito IS NOT NULL
GROUP BY departamento, provincia, distrito
ORDER BY total_damnificados DESC, total_eventos DESC
LIMIT 10;